### MNIST Binary Classifier

In this notebook, we will implement a CNN classifier to classify the digits 0 and 1 from the MNIST dataset. The objective of this lesson is twofold:
* To build our first CNN classifier (binary).
* To demonstrate the importance of cross-entropy.

Let's start with the usual imports.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

from time import time
import numpy as np
import keras
from keras import Model
from keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

# Set the seed for reproducibility (python, numpy and the backend)
keras.utils.set_random_seed(1234)

#### Dataset Loading

We have already inspected the MNIST dataset. We are going to load it now since we are going to use it for training the classifier.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Dataset params
size = x_train.shape[1]

print('Train set:   ', len(y_train), 'samples')
print('Test set:    ', len(y_test), 'samples')
print('Sample dims: ', x_train.shape)

#### Dataset Preprocessing

In this example, we are going to train a binary classifier to classify the digits 0 and 1. Therefore, we have to remove all other digits (classes) from the dataset.

In [ ]:
mask_train = np.logical_or(y_train == 0, y_train == 1)
x_train = x_train[mask_train, ...]
y_train = y_train[mask_train]

mask_test = np.logical_or(y_test == 0, y_test == 1)
x_test = x_test[mask_test, ...]
y_test = y_test[mask_test]

# Normalization
x_train = x_train/255
x_test = x_test/255

print('Train set:   ', len(y_train), 'samples')
print('Test set:    ', len(y_test), 'samples')
print('Sample dims: ', x_train.shape)

#### Building the Classifier

We are going to build a relatively simple convolutional neural network (CNN) for this task. It stacks two blocks of convolution + max pooling, then flattens the result and feeds it to a single output neuron.

In [ ]:
inputs = Input(shape=(size, size, 1))

net = Conv2D(16, kernel_size=(3, 3), activation="relu")(inputs)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Conv2D(32, kernel_size=(3, 3), activation="relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Flatten()(net)
outputs = Dense(1, activation="linear")(net)

model = Model(inputs, outputs)
model.summary()

Let's follow the tensor through the network (and compare it with the summary above):

| Layer | Output shape | Parameters |
|---|---|---|
| Input | 28x28x1 | 0 |
| Conv2D (16 filters 3x3) | 26x26x16 | 16 x (3x3x1) + 16 biases = 160 |
| MaxPooling2D (2x2) | 13x13x16 | 0 |
| Conv2D (32 filters 3x3) | 11x11x32 | 32 x (3x3x16) + 32 biases = 4,640 |
| MaxPooling2D (2x2) | 5x5x32 | 0 |
| Flatten | 800 | 0 |
| Dense (1) | 1 | 800 + 1 bias = 801 |

The convolutions have no padding, so each one shrinks the image by 2 pixels, and each pooling halves the spatial size. Note that the convolutional layers have very few parameters because the same small filters are shared over the entire image. This is an extremely simple model (for a usual classification task) with just 5,601 (trainable) parameters. That is still far less than the 12.5k parameters we used in the previous lesson, where a fully connected network was applied to the flattened image.

In [ ]:
# Plot the model
keras.utils.plot_model(model, to_file="model.png", show_shapes=True, show_layer_names=True)

#### Training

Let's now compile and train the model. We will use the well-known MSE as our loss function.

Note: MSE is **not** the suitable loss for classification task but it serves us here well for the demonstration purposes. We will learn how to design a classifier in a proper way later in this lesson ;-)

In [ ]:
epochs = 25
batch_size = 128

model.compile(loss="mse", optimizer="adam", metrics=["accuracy"])

start = time()
history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1, verbose=2)
print('Elapsed time', time() - start)

Let's now plot the history to see the evolution of the training.

In [ ]:
def plot_history(history):
    h = history.history
    epochs = range(len(h['loss']))

    plt.subplot(121), plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
    plt.legend(['Train', 'Validation'])
    plt.subplot(122), plt.plot(epochs, h['accuracy'], '.-',
                               epochs, h['val_accuracy'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Accuracy')
    plt.legend(['Train', 'Validation'])
        
    print('Train Acc     ', h['accuracy'][-1])
    print('Validation Acc', h['val_accuracy'][-1])
    
plot_history(history)

#### Evaluation

As we did in the previous lesson, we are going to evaluate the trained model.

In [ ]:
# Compute the predictions on the test set
y_pred = model.predict(x_test, verbose=0)

print('True', y_test[0:5].flatten())
print('Pred', y_pred[0:5].flatten())

In [ ]:
y_true = y_test.flatten()
y_pred = y_pred.flatten() > 0.5

# Overall accuracy
num_samples = len(y_true)
acc = np.sum(y_test == y_pred)/num_samples

# Accuracy for digit 0
mask = y_true == 0
acc0 = np.sum(y_test[mask] == y_pred[mask])/np.sum(mask)

# Accuracy for digit 1
mask = y_true == 1
acc1 = np.sum(y_test[mask] == y_pred[mask])/np.sum(mask)

print('Overall acc', acc)
print('Digit-0 acc', acc0)
print('Digit-1 acc', acc1)

We now visualise some of the evaluation results.

In [ ]:
for ii in range(15):
    idx = np.random.randint(0, len(y_pred))
    plt.subplot(3,5,ii+1), plt.imshow(x_test[idx, ...], cmap='gray')
    plt.title('True: ' + str(y_true[idx]) + ' | Pred: ' + str(int(y_pred[idx])))

#### What Has the Network Learnt?

In the Sobel notebook, we trained a single 3x3 filter to imitate a filter that was designed by hand. Here, nobody told the network which filters to use: the 16 filters of the first convolutional layer were learnt from the data, in order to tell zeros from ones. Let's look at them.

In [ ]:
conv1, pool1 = model.layers[1], model.layers[2]

kernels = conv1.get_weights()[0]    # shape: (3, 3, 1, 16)

for ii in range(16):
    plt.subplot(4, 4, ii+1), plt.imshow(kernels[:, :, 0, ii], cmap='gray')
    plt.title('Filter ' + str(ii)), plt.axis('off')

Unlike the Sobel filter, these filters are not something we could have easily designed by hand, and being only 3x3 they are hard to interpret on their own. It is more instructive to see what they do. Let's apply the first convolution (and the first pooling) to a test image and display the 16 resulting feature maps, one per filter.

In [ ]:
# Sub-model that returns the outputs of the first convolution and of the first pooling
intermediate = Model(model.inputs, [conv1.output, pool1.output])

# Take the first zero from the test set
idx = np.where(y_test == 0)[0][0]
conv_maps, pool_maps = intermediate.predict(x_test[idx:idx+1], verbose=0)
conv_maps, pool_maps = conv_maps[0], pool_maps[0]    # remove the batch dimension
print('Conv output:', conv_maps.shape)
print('Pool output:', pool_maps.shape)

fig, axs = plt.subplots(3, 16, figsize=(16, 3.5))
for ax in axs.flatten():
    ax.axis('off')
axs[0, 0].imshow(x_test[idx], cmap='gray'), axs[0, 0].set_title('Input')
for ii in range(16):
    axs[1, ii].imshow(conv_maps[:, :, ii], cmap='gray')
    axs[2, ii].imshow(pool_maps[:, :, ii], cmap='gray')
axs[1, 0].set_title('After Conv2D (26x26)', loc='left')
axs[2, 0].set_title('After MaxPooling2D (13x13)', loc='left')

Each feature map is the response of one filter at every position of the image. Some maps light up on the stroke of the digit, some on the background, and some only on parts of its contour. The last row shows the effect of max pooling: the maps are half the size (13x13) but the strongest responses are kept, so the shape of the digit is still clearly visible. This is how the combination of convolution and pooling distills the features: the next convolutional layer will work on these 16 smaller maps. A smaller size also means less computation in the following layers.

### Full MNIST Classification

So far we have implemented a binary MNIST classifier that is able to classify handwritten zeros and ones. However, the MNIST dataset contains (obviously) more digits. How can we extend the classifier to account for all possible digits? Would a straightforward extension work? Let's see :-)

First, we are going to load the data (and keep all the digits!) and build the exactly same CNN as before.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalization
x_train = x_train/255
x_test = x_test/255

print('Train set:   ', len(y_train), 'samples')
print('Test set:    ', len(y_test), 'samples')
print('Sample dims: ', x_train.shape)

In [ ]:
inputs = Input(shape=(size, size, 1))

net = Conv2D(16, kernel_size=(3, 3), activation="relu")(inputs)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Conv2D(32, kernel_size=(3, 3), activation="relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Flatten()(net)
outputs = Dense(1, activation="linear")(net)

model = Model(inputs, outputs)
model.summary()

#### Training

We train the network the same way we did before. We are going to use the MSE as the loss function and demonstrate why it is not a good idea to use it for classification purposes :-) This is also the reason why we cannot use the built-in accuracy metric anymore.

Also note that the training now takes longer since we are using all the available data (we didn't filter out any samples).

In [ ]:
epochs = 25
batch_size = 128

model.compile(loss="mse", optimizer="adam", metrics=["mse"])

start = time()
history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1, verbose=2)
print('Elapsed time', time() - start)

In [ ]:
h = history.history
epochs = range(len(h['loss']))

plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
plt.legend(['Train', 'Validation'])

#### Evaluation

Now we are going to evaluate the classifier accuracy for all digits.

In [ ]:
y_pred = model.predict(x_test, verbose=0)

print('True', y_test[100:105].flatten())
print('Pred', y_pred[100:105].flatten())

In [ ]:
y_true = y_test.flatten()
y_pred = y_pred.flatten()
digits = range(0, 10)

for digit in digits:
    mask = y_true == digit
    # Count the true positives (the closest digit)
    tp = np.sum(np.abs(y_pred[mask] - digit) < 0.5)
    total = np.sum(mask)
    print('Digit-', digit, ' acc', tp/total)

print('Overall acc', np.mean(np.abs(y_pred - y_true) < 0.5))

We see that the accuracy is actually quite bad (only about half of the test digits are within 0.5 of the true value) and the predictions are not close enough to the ground truth. The reason is that we are using an inappropriate loss function for the classification task.

Regressing the digit as a single number imposes an order and a distance between the classes. For the MSE, predicting 5 for a 4 is a small error and predicting 9 for a 4 is a big one, as if the digits were points on a line. But the classes have no such order: a 4 may look like a 9 and not at all like a 5. The network is forced to arrange all the digits on a line, which is a very hard thing to do.

The plot below shows the distribution of the predicted values for each true digit. The medians roughly follow the ideal line, so the network did learn something. However, the predictions are widely spread, the boxes of neighbouring digits touch or overlap, and there are outliers far away from the correct value (e.g., some nines are predicted well below 6).

In [ ]:
# Distribution of the predictions for each true digit
plt.boxplot([y_pred[y_true == digit] for digit in digits], positions=digits)
plt.plot(digits, digits, 'r--')
plt.grid(True), plt.xlabel('true digit'), plt.ylabel('predicted value')
plt.legend(['perfect prediction'])

The right way of handling multiple classes is to give each class its own output and to use a loss function that compares probability distributions. We do that in the next notebook using one-hot encoding, the softmax activation and the cross-entropy loss.